# 텍스트 정제와 정규화

텍스트 전처리는 원문을 모델이 다루기 쉬운 형태로 바꾸는 과정이다. 이때 **정규화(normalization)** 는 같은 의미로 취급할 여러 표기를 하나의 기준형으로 맞추는 작업이고,
**정제(cleansing 또는 cleaning)** 는 과업에 불필요하거나 잘못되었거나 민감한 정보를 제거·필터링·마스킹하는 작업이다.
이 노트북에서는 사용한 함수 이름이 아니라 **처리 목적과 보존되는 정보**를 기준으로 두 작업을 구분하고, 각 코드의 변환 전후를 실제 출력으로 확인한다.

## 정규화와 정제의 정의 및 구분 기준

### 정규화(normalization)

정규화는 **분석 목적상 같은 것으로 취급하기로 한 여러 표면형을 하나의 기준 표현으로 맞추는 작업**이다.
표면형은 화면에 실제로 나타난 문자열 형태를 뜻한다. 정규화의 핵심 질문은 ‘표현 방식만 달랐던 값을 같은 기준으로 비교할 수 있게 만들었는가?’이다.

- `United Kingdom → UK`: 같은 국가명을 기준 용어로 통합하는 정규화이다.
- `Theater → theater`: 대소문자 차이를 무시하기로 한 과업의 정규화이다.
- `ＡＩ → AI`, 연속 공백을 한 칸으로 통합: 문자 표현 형식을 맞추는 정규화이다.

정규화도 상황에 따라 원래 의미의 구분을 잃게 할 수 있다.
개체명 인식에서 대소문자를 없애거나 상품 코드의 호환 문자를 합치면 필요한 구분이 사라질 수 있으므로, 무엇을 같은 표현으로 볼 것인지 과업별로 결정해야 한다.

### 정제(cleansing 또는 cleaning)

정제는 **현재 과업에 불필요하거나 잘못되었거나 민감하다고 판단한 정보를 제거·필터링·마스킹하는 작업**이다.
정제의 핵심 질문은 ‘모델에 넣지 않을 정보를 줄이거나 안전한 값으로 가렸는가?’이다.

- 희귀 토큰·짧은 토큰·불용어 제거: 학습에 기여가 낮다고 판단한 정보를 제외하는 정제이다.
- `master@example.com → __EMAIL__`: 민감한 실제 값을 가리면서 이메일이 있었다는 사실은 남기는 정제이다.
- HTML 태그나 깨진 제어 문자 제거: 본문 분석에 불필요한 노이즈를 없애는 정제이다.

정제는 정보를 실제로 줄이거나 가리므로 손실 위험이 더 직접적이다. 감성 분석의 `not`, 기술 문서의 `AI`, 보안 로그의 URL처럼 과업에 중요한 값을 제거하지 않았는지 전후 샘플로 확인해야 한다.

### 구분할 때 기억할 기준

| 판단 질문 | 해당 처리 | 예시 |
| --- | --- | --- |
| 같은 것으로 볼 표현을 기준형으로 맞추는가? | 정규화 | `Straße`와 `STRASSE`를 비교 가능한 형태로 통합 |
| 넣지 않을 정보를 제거하거나 가리는가? | 정제 | 불용어 제거, 이메일 마스킹 |
| 문장을 토큰이나 형태소로 나누는가? | 둘 다 아님 | 토큰화, Okt 형태소 분석 |
| 여러 목적을 한 함수에서 연속 수행하는가? | 복합 처리 | 정규화 후 마스킹과 불용어 제거 |

### 경계 사례와 복합 처리

하나의 코드셀에 정규화·토큰화·정제가 연속해서 들어갈 수 있다. 이때 셀 전체를 한 단어로만 부르지 않고 각 연산의 목적을 분리해 설명한다.

- `text.casefold()` 뒤에 희귀 토큰을 제거하면 **대소문자 정규화 후 희귀 토큰 정제**를 수행한 복합 처리이다.
- 짧은 단어를 삭제한 뒤 연속 공백을 한 칸으로 줄이면 **짧은 단어 정제 후 공백 정규화**를 수행한 복합 처리이다.
- 이메일을 `__EMAIL__`로 바꾸면 여러 주소가 같은 표기로 통합된다는 점에서는 정규화 성격도 있지만, 이 교안에서는 실제 개인정보를 숨기는 목적이 우선이므로 **민감 정보 정제**로 분류한다.
- 형태소 분석은 문장을 분석 단위로 나누는 **선행 단계**이고, 그 결과에서 조사나 불용어를 제거할 때 비로소 정제가 일어난다.

### 코드에서 분류하는 순서

1. 입력에서 무엇이 바뀌거나 사라지는지 확인한다.
2. 의미상 같은 표면형을 기준형으로 맞추면 정규화로 판단한다.
3. 정보 자체를 제거하거나 가리면 정제로 판단한다.
4. 문장을 토큰이나 형태소로 나누기만 하면 선행 단계로 판단한다.
5. 둘 이상의 목적이 연속되면 복합 처리로 표시하고 각 단계의 입력과 출력을 따로 확인한다.

> **중요:** 사용한 도구만으로 분류하지 않는다. `re.sub`로 `United Kingdom`을 `UK`로 통합하면 정규화이고, 이메일을 `__EMAIL__`로 가리면 이 교안의 분류 기준에서는 개인정보 정제이다. 같은 함수도 목적과 보존하려는 정보에 따라 분류가 달라질 수 있다.


## 실습 환경 준비

정규화와 정제 실습에서는 정규식, 유니코드 처리, 단어 빈도 계산과 NLTK 토큰화·불용어 사전을 사용한다. 다음 셀은 필요한 모듈을 직접 불러오고 NLTK 데이터 리소스를 내려받는다. `punkt`와 `punkt_tab`은 영어 토큰화에 사용하고, `stopwords`는 영어 불용어 목록을 제공한다.


In [1]:
import re
import unicodedata
from collections import Counter
from pathlib import Path

import nltk
from nltk.tokenize import word_tokenize
from pandas.core.interchange.from_dataframe import set_nulls
from pandas.io.sas.sas_constants import encoding_length
from prompt_toolkit.styles import default_pygments_style

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

## 처리 순서를 먼저 정하는 이유

권장 출발점은 **원문 보존 → 유니코드·도메인 표기 정규화 → 민감 정보 마스킹 → 토큰화 → 과업별 필터링 → 전후 검증** 순서이다.
표기를 먼저 통합해야 같은 단어의 빈도가 하나로 집계되고, 이메일과 URL은 토큰화 전에 마스킹해야 여러 조각으로 흩어지는 것을 막을 수 있다.

다만 하나의 순서가 모든 과업에 정답은 아니다. 감성 분석에서는 느낌표와 부정어가 신호이고, 개체명 인식에서는 대소문자가 신호이며, 보안 로그에서는 URL과 숫자가 핵심일 수 있다.
따라서 규칙은 도메인 요구사항으로 관리하고 학습 데이터와 실제 추론 입력에 동일하게 적용해야 한다.


## 도메인 용어를 하나의 표기로 정규화

용어 정규화는 서로 다른 문자열이 같은 대상을 뜻한다는 도메인 지식을 코드로 표현한다.
`re.escape`는 사전의 원문 표기에 정규식 특수문자가 있어도 문자 그대로 찾게 하고, `re.IGNORECASE`는 대소문자 차이를 무시한다. 함수는 원문을 직접 바꾸지 않고 기준 표기로 통합한 새 문자열을 반환한다.


In [3]:
# 원문과 기준 표현을 전달 받아 치환(정규화) 결과를 담은 새 문자열을 반환
def normalize_terms(text, mapping):
    ordered_terms = sorted(
        mapping.items(), # (찾을 표면형, 기준 표현) 튜플을 하나씩 아래 key 에 제공
        key = lambda item : len(item[0]), # 찾을 표면형의 길이를 반환
        reverse = True # 정렬 시 길이가 긴 문자를 앞으로 배치(내림차순)
    )

    # normalized : 매 단계의 치환(정규화) 결과를 저장할 작업용 변수
    normalized = text

    # \b문자열\b : 긴 문자열 내부가 아닌 독립된 단어 경계를 지정
    for source, target in ordered_terms:
        pattern = rf"\b{re.escape(source)}\b"

        # re.escape("\n): . \ () 등의 역할 대신 일반 문자로 인식시킴
        # re.sub() : 패턴이 일치한 모든 부분을 target으로 변환
        # re.IGNORECASE : 대소문자 구분 X

        normalized = re.sub(
            pattern,
            target,
            normalized,
            flags = re.IGNORECASE
            )
    return normalized

raw_text = "The United Kingdom and UK have a long history together."

term_mapping = {
    "United Kingdom": "UK",
}

normalized_text = normalize_terms(raw_text, term_mapping)

print("원문: ", raw_text)
print("정규화: ", normalized_text)
# 원하는 결과
# 원문: The United Kingdom and UK have a long history together.
# 정규화: The UK and UK have a long history together.

원문:  The United Kingdom and UK have a long history together.
정규화:  The UK and UK have a long history together.


## 대소문자 정규화와 casefold

`str.lower`는 일반적인 소문자 변환이고 `str.casefold`는 국제 문자 비교에 더 강한 변환이다.
예를 들어 독일어 `ß`는 `lower`에서 유지되지만 `casefold`에서는 `ss`로 통합된다.
두 메서드는 원문을 변경하지 않고 새 문자열을 반환한다. 서로 다른 표기를 비교 가능한 기준형으로 맞추므로 정규화이지만, 대소문자와 문자 차이가 실제 의미일 때는 정보 손실이 된다.


In [5]:
raw_text = (
    "Theater attendance has decreased. "
    "The THEATER industry compares Straße and STRASSE."
)

term_maaping = {
    "United Kingdom" : "UK"
}

lower_text = raw_text.lower()
print("lower: ", lower_text)

casefold_text = raw_text.lower()
print("casefold: ", casefold_text)


lower:  theater attendance has decreased. the theater industry compares straße and strasse.
casefold:  theater attendance has decreased. the theater industry compares straße and strasse.


## 유니코드와 공백 정규화

화면에서 비슷해 보이는 글자도 내부 유니코드 코드가 다르면 서로 다른 토큰으로 집계될 수 있다.
`unicodedata.normalize`의 `NFC`는 정준적으로 같은 문자를 표준 조합 형태로 맞추고, `NFKC`는 전각 문자와 호환 문자까지 더 적극적으로 통합한다.
`re.sub(r"\s+", " ", ...)`는 탭과 연속 공백을 한 칸으로 맞춘다. 모두 표현 형식을 기준형으로 맞추므로 정규화이다.


In [8]:
raw_text = "ＡＩ   모델\t가격은 ① 10,000원이다."

# NFD ( ㄱ + ㅏ + ㅇ) 형태의 글자를 NFC(강) 형태로 통일
# ex ) Mac 한글명 파일 -> Windows 로 이동 시 한글이 쪼개지는 문제
nfc_text = unicodedata.normalize("NFC", raw_text)

nfkc_text = unicodedata.normalize("NFKC", raw_text)

# 연속되는 공백 문자를 한 칸 띄어쓰기로 변경
# 문자열.strip(): 양 옆 공백 제거
nfkc_and_space_text = re.sub(r"\s+", " ", nfkc_text).strip()

print("원문: ", raw_text)
print("NFC: ", nfc_text)
print("NFKC: ", nfkc_text)
print("공백 제거", nfkc_and_space_text)

원문:  ＡＩ   모델	가격은 ① 10,000원이다.
NFC:  ＡＩ   모델	가격은 ① 10,000원이다.
NFKC:  AI   모델	가격은 1 10,000원이다.
공백 제거 AI 모델 가격은 1 10,000원이다.


## 빈도가 낮은 단어 필터링

`Counter`는 각 토큰의 등장 횟수를 계산한다. 먼저 `casefold`로 `The`와 `the`를 같은 표기로 맞춘 뒤, 토큰화하고 `isalpha()`가 거짓인 구두점을 제외한다. 마지막으로 최소 빈도보다 작은 토큰을 제거한다. 따라서 이 셀은 정규화·선행 단계·정제가 연속된 예제이다. 빈도 필터는 `Counter`의 고유 키만 반환하지 않고 원래 토큰 목록을 순회해야 중복과 순서가 보존된다.

실무에서는 이 기준을 한 문장이 아니라 **학습 코퍼스 전체**에서 계산해야 한다. 검증·테스트 데이터까지 사용해 빈도 사전을 만들면 미래 데이터 정보가 섞이므로 학습 데이터에서만 기준을 학습한다.


In [13]:
text = (
    "The quick brown fox jumps over the lazy dog, "
    "while the fox is quick and agile."
)

# 문아열 case 맞추기
normalized_text = text.casefold()

# 토큰화
# preserve_line = True : 입력된 텍스트를 한 줄로 취급
tokens = word_tokenize(normalized_text, preserve_line=True)

# 단어 토큰만 추출
word_tokens = [token for token in tokens if token.isalpha()]

# 같은 단어 빈도 측정(count)
# Counter(list) : 같은 값을 가지는 요소의 개수를 세서 반환
# key : 각 요소(토큰)
# value : 카운트 수
word_counter = Counter(word_tokens)
print(word_counter)

# min_count 미만의 토큰을 제거한 리스트를 생성

min_count =2

removed_tokens = [
    token for token in word_tokens if word_counter[token] < min_count
]

# min_count 이상의 토큰 리스트를 생성 (필터링 완료된 토큰 리스트)
filtered_tokens = [
    token for token in word_tokens if word_counter[token] >= min_count
]

print("제거된 토큰: ", removed_tokens)
print("2회 이상 토큰: ", filtered_tokens)

Counter({'the': 3, 'quick': 2, 'fox': 2, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1, 'while': 1, 'is': 1, 'and': 1, 'agile': 1})
제거된 토큰:  ['brown', 'jumps', 'over', 'lazy', 'dog', 'while', 'is', 'and', 'agile']
2회 이상 토큰:  ['the', 'quick', 'fox', 'the', 'the', 'fox', 'quick']


## 길이가 짧은 단어 필터링

`len` 기반 필터는 토큰의 문자 수만 보고 제거 여부를 정하므로 정제이다. 영어의 `a`, `is`, `on`뿐 아니라 `AI`, `UK`, `R`과 같은 핵심 토큰도 함께 제거할 수 있다. 토큰화와 구두점 제외는 필터 기준을 적용하기 위한 선행 처리이며, 핵심 정제 기준은 `len(token) < 3`이다.


In [14]:
text = "A fox is on the run in the forest and AI can help."

tokens = [
    token
    for token in word_tokenize(text, preserve_line=True)
    if token.isalpha()
]

min_length = 3 # 최소 글자 길이

removed_tokens = [
    token
    for token in tokens
        if len(token) < min_length
]

filtered_tokens = [
    token
    for token in tokens
        if len(token) >= min_length
]


print("원래 단어: ", tokens)
print("제거된 단어:", removed_tokens)
print("남은 단어:", filtered_tokens)

원래 단어:  ['A', 'fox', 'is', 'on', 'the', 'run', 'in', 'the', 'forest', 'and', 'AI', 'can', 'help']
제거된 단어: ['A', 'is', 'on', 'in', 'AI']
남은 단어: ['fox', 'the', 'run', 'the', 'forest', 'and', 'can', 'help']


## 정규식으로 짧은 영단어 제거

`re.compile`은 반복해서 사용할 정규식 패턴 객체를 만든다. `\b[A-Za-z]{1,2}\b`는 단어 경계 사이의 1~2글자 영단어를 찾고, `Pattern.sub`는 일치 부분을 빈 문자열로 바꾸어 제거한다. 삭제로 생긴 연속 공백을 `re.sub(r"\s+", " ", ...)`로 한 칸에 맞추므로 한 셀 안에 정제와 정규화가 모두 있다.

이 패턴은 영문만 대상으로 한다. 숫자·한글·혼합 코드를 같은 규칙으로 처리하려면 도메인 요구에 맞는 별도 패턴이 필요하다.


In [15]:
text = "A fox is on the run in the forest and AI can help."

short_word_pattern = re.compile(r"\b[A-Za-z]{1,2}\b")
matched_short_words = short_word_pattern.findall(text)
removed_text = short_word_pattern.sub("", text)

cleaned_text = re.sub(r"\s+", " ", removed_text).strip()

print(f"패턴 자료형: {type(short_word_pattern).__name__}")
print(f"정규식 일치: {matched_short_words}")
print(f"정제 결과: {cleaned_text}")


패턴 자료형: Pattern
정규식 일치: ['A', 'is', 'on', 'in', 'AI']
정제 결과: fox the run the forest and can help.


## 이메일과 URL 마스킹

개인정보와 URL은 삭제할 수도 있지만, 완전히 지우면 원래 위치와 데이터 품질 문제를 추적하기 어렵다.
이메일과 URL을 고정 토큰으로 마스킹해 민감한 값은 숨기면서 해당 정보가 있었다는 사실은 보존한다.
`Pattern.sub`의 입력은 대체 문자열과 원문이고 반환값은 마스킹된 새 문자열이다.
실제 값을 숨기는 목적이므로 정제로 분류한다.

정규식은 모든 실제 이메일과 URL 문법을 완벽하게 처리하지 못한다. 배포 전에는 도메인의 정상·오류 샘플을 테스트하고, HTML 본문은 정규식보다 HTML 파서를 우선한다.


In [16]:
email_pattern = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

url_pattern = re.compile(r"https?://[^\s!?]+")

def mask_sensitive_text(text):
    masked = email_pattern.sub("__EMAIL__", text)
    masked = url_pattern.sub("__URL__", masked)
    return masked

raw_text = (
    "Contact us at master@lifeisgood.com for more details. "
    "Visit our site at https://lifeisgood.com!!!"
)
masked_text = mask_sensitive_text(raw_text)

print(f"원문: {raw_text}")
print(f"마스킹: {masked_text}")


원문: Contact us at master@lifeisgood.com for more details. Visit our site at https://lifeisgood.com!!!
마스킹: Contact us at __EMAIL__ for more details. Visit our site at __URL__!!!


## 영어 불용어 사전 적용

불용어(stopword)는 문장에 자주 나타나지만 현재 과업에서 정보량이 낮다고 판단해 제외하는 단어이다.
NLTK의 `stopwords.words("english")`는 영어 불용어 목록을 반환하고, `set`으로 바꾸면 각 토큰의 포함 여부를 빠르게 검사할 수 있다.

토큰별 `casefold`로 대소문자를 정규화하고, 토큰화한 뒤 구두점과 불용어를 정제한다.
출력에서 실제로 제거된 기능어와 남은 내용어를 비교하고, 중요한 단어가 사전에 포함되지 않았는지 확인한다.


In [17]:
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english"))

text = "She sells seashells by the seashore, and it is a beautiful day."


print(len(english_stopwords))
print(english_stopwords)

198
{'here', 'against', 'if', 'how', 'about', 'down', 'itself', 'she', 'other', 'should', 'his', 'as', "shouldn't", 'your', 'with', 'm', 'weren', "we'd", 'you', 'each', 'at', 'only', 've', 'such', 'or', 'our', 'd', "we're", 'same', "she'd", "should've", "didn't", 'after', "doesn't", 'yourself', 'hasn', "they'll", 'we', 'am', 'isn', 'nor', 'few', "you'd", 'herself', 'not', "i'll", 'they', 'until', 'is', 'be', 'being', 'who', 'by', 'll', 'further', 'that', 'won', 'it', 'while', "we'll", 'now', "they're", 'them', "i'm", "aren't", 'hers', "don't", 'once', 'ain', 'doesn', 'into', 'very', 'below', "it'll", "won't", 'where', 'between', 'has', 'can', "hadn't", 'the', 'some', "it's", "he'll", 'then', "they'd", 'again', 'so', 'because', 'an', 'were', 'what', 'aren', 'this', 'ourselves', 'didn', 'their', 'for', 'in', "shan't", 'myself', 'from', 'theirs', 's', 'couldn', 'had', "he'd", 'ours', 'any', 'and', 'why', 'yourselves', 'those', 'ma', 'will', "wouldn't", 'whom', "you'll", "we've", 'he', 'ar

## 부정어를 보호해야 하는 감성 분석

영어 기본 불용어에는 `not`, `no`, `nor` 같은 부정어가 포함될 수 있다.
감성 분석에서 `not`을 제거하면 `not good`이 `good`과 비슷한 입력이 되어 의미가 뒤집힌다.
집합 차집합으로 부정어 세 개를 불용어 사전에서 제외하면 나머지 기능어는 제거하면서 부정 의미는 보존할 수 있다.


In [18]:
text = "The service is not good."

# 토큰화
tokens = [
    token.casefold()
    for token in word_tokenize(text, preserve_line=True)
    if token.isalpha()
]

# 불용어 사전에 포함된 토큰 찾기
removed_tokens = [
    token
    for token in tokens
    if token in english_stopwords # 포함 여부 확인
]

# 불용어 사전에 포함되지 않는 토큰 찾기
filtered_tokens = [
    token
    for token in tokens
    if token not in english_stopwords # 포함 X 확인
]


print("불용어 수 :", len(english_stopwords))
print("제거된 토큰 :", removed_tokens)
print("남은 토큰 :", filtered_tokens)


불용어 수 : 198
제거된 토큰 : ['the', 'is', 'not']
남은 토큰 : ['service', 'good']


In [22]:
text = "The service is not good."

tokens = [
    token.casefold()
    for token in word_tokenize(text, preserve_line=True)
    if token.isalpha()
]

# english_stopwords(기본 영어 불용어 사전)을 그대로 적용
default_filtered = [
    token
    for token in tokens
    if token not in english_stopwords
]

print("원래 토큰: ", tokens)
print("기본 불용어 적용:  ", default_filtered)

# 감성 분석에 필요한 부정어 집합(set) 생성
protected_negations = {"not", "no", "nor"}

# 감성 분석용 불용어 사전(커스텀)
# set 끼리 - 연산(차집합) 가능
sentiment_stopwords = english_stopwords - protected_negations

sentiment_filtered = [
    token
    for token in tokens
    if token not in sentiment_stopwords
]

print("부정어 보호 적용 : ", sentiment_filtered)

원래 토큰:  ['the', 'service', 'is', 'not', 'good']
기본 불용어 적용:   ['service', 'good']
부정어 보호 적용 :  ['service', 'not', 'good']


## KoNLPy 설치와 Okt 불러오기

한국어는 조사와 어미가 단어에 붙으므로 공백 분리만으로 불용어 단위를 맞추기 어렵다.
KoNLPy의 `Okt`는 한국어 문장을 형태소 토큰 목록으로 나누는 분석기이다. 형태소 분석 자체는 정제나 정규화가 아니라, 이후 불용어를 같은 단위로 비교하기 위한 선행 단계이다.

KoNLPy를 현재 커널 환경에 설치하고 `Okt`를 직접 불러온다.


In [23]:
%pip install -q konlpy

Note: you may need to restart the kernel to use updated packages.


### Okt(Open Korea Text) 클래스 불러오기

설치된 KoNLPy에서 `Okt` 클래스만 불러온다. `from konlpy.tag import Okt`가 성공하면 뒤에서 `Okt()`로 형태소 분석기 객체를 만들 수 있다. 이 셀은 분석기를 준비하는 단계이며 실제 형태소 분리는 뒤의 `morphs()` 호출에서 수행한다.


In [27]:
from konlpy.tag import Okt

%conda install -c conda-forge openjdk=17

import os
import sys

os.environ["JAVA_HOME"] = sys.prefix
os.environ["PATH"] = f"{sys.prefix}/bin:{os.environ['PATH']}"


Channels:
 - conda-forge
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: C:\Users\playdata2\miniforge3\envs\nlp_env

  added / updated specs:
    - openjdk=17


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    openjdk-17.0.18            |       he453025_0       161.1 MB  conda-forge
    symlink-exe-runtime-1.0    |       hfd05255_2          13 KB  conda-forge
    ------------------------------------------------------------
                                           Total:       161.1 MB

The following NEW packages will be INSTALLED:

  openjdk            conda-forge/win-64::openjdk-17.0.18-he453025_0 
  symlink-exe-runti~ conda-forge/win-64::symlink-exe-runtime-1.0-hfd05255_2 



openjdk-17.0.18      | 161.1 MB  |            |   0% 

symlink-exe-runtime- | 13 KB     |            |   0% 

symlink-exe-runtime- | 13 KB     | ########## | 100% 

symlink

## 한국어 형태소 단위 불용어 비교

이 셀은 Okt 설치 여부와 관계없이 형태소 분석 결과라고 가정한 작은 토큰 목록으로 불용어 필터의 원리를 확인한다. `set` 포함 검사는 토큰 문자열이 불용어 또는 구두점 사전과 정확히 같을 때만 제거한다. 실제 프로젝트에서는 분석기 출력 단위와 사전 항목 단위가 같은지 먼저 확인해야 한다.


In [28]:
morpheme_tokens = [
    "이", "방법", "은", "특히", "빅데이터", "에서",
    "중요한", "역할", "을", "합니다", ".",
    "이를", "통해", "더", "많은", "정보", "를",
    "얻을", "수", "있습니다", ".",
]

korean_stopwords = {
    "이", "은", "에서", "을", "를", "합니다",
    "이를", "통해", "더", "수", "있습니다",
}
punctuation = {"."}


filtered_tokens = [
    token
    for token in tokens
    if token not in korean_stopwords and token not in punctuation
]

## Okt 형태소 분석과 불용어 제거

`Okt.morphs`는 한국어 문자열을 형태소 토큰 목록으로 반환한다. 먼저 얻은 `okt_tokens`는 정제를 적용하기 전의 선행 결과이고, 불용어와 구두점을 제외한 `okt_filtered_tokens`가 정제 결과이다.

형태소 분석기의 버전과 사전이 바뀌면 토큰 경계도 달라질 수 있다. 실행 결과에서는 형태소 목록과 정제 목록을 나란히 비교해 조사·어미가 제거되는지, `방법`, `빅데이터`, `역할`, `정보` 같은 내용어가 남는지 확인한다.


In [32]:
korean_text = (
    "이 방법은 특히 빅데이터에서 중요한 역할을 합니다. "
    "이를 통해 더 많은 정보를 얻을 수 있습니다."
)

# Okt에서 제공하는 형태소 분석 기능을 가진 객체 Okt() 생성
okt = Okt()

# 원문을 입력 받아 원문 순서의 형태소 리스트를 반환
okt_tokens = okt.morphs(korean_text)

print("Okt 형태소:", okt_tokens)


korean_stopwords = {
    "이", "은", "에서", "을", "를", "합니다",
    "이를", "통해", "더", "수", "있습니다",
}

punctuation = {"."}

filtered_tokens = [
    token
    for token in okt_tokens
    if token not in korean_stopwords and token not in punctuation
]

print("Okt 불용어 제거: ", filtered_tokens)

Okt 형태소: ['이', '방법', '은', '특히', '빅데이터', '에서', '중요한', '역할', '을', '합니다', '.', '이를', '통해', '더', '많은', '정보', '를', '얻을', '수', '있습니다', '.']
Okt 불용어 제거:  ['방법', '특히', '빅데이터', '중요한', '역할', '많은', '정보', '얻을']


## 다어절 불용어의 토큰 경계

`이를 통해`처럼 두 토큰으로 분리되는 표현은 문자열 하나인 `"이를 통해"`와 직접 일치하지 않는다. 단순 `set` 조회는 토큰 하나씩만 비교하므로 제거에 실패한다. `remove_token_phrases`는 현재 위치에서 각 토큰 튜플 길이만큼 슬라이싱해 연속 구간이 같은지 비교하고, 일치하면 그 길이만큼 건너뛴다.


In [33]:
tokens = ["이를", "통해", "더", "많은", "정보를", "얻는다"]

naive_stopwords = {"이를 통해"}

naive_filtered = [
    token
    for token in tokens
    if token not in naive_stopwords
]

def remove_token_phrases(tokens, phrases):
    ordered_phrases = sorted(
        phrases,
        key=len,
        reverse=True,
    )


    result = []
    index = 0

    while index < len(tokens):
        matched_phrase = next(
            (
                phrase
                for phrase in ordered_phrases
                if tokens[index:index + len(phrase)] == list(phrase)
            ),
            None,
        )

        # 분기 결과
        # - None이면 현재 토큰을 보존하고 index를 1 증가시킨다.
        # - 구문이 있으면 구문을 결과에 넣지 않고 index를 구문 길이만큼 증가시킨다.
        if matched_phrase is None:
            result.append(tokens[index])
            index += 1
        else:
            index += len(matched_phrase)

    return result

phrase_filtered = remove_token_phrases(
    tokens,
    phrases=[("이를", "통해")],
)

print(f"단순 토큰 비교: {naive_filtered}")
print(f"다어절 구문 비교: {phrase_filtered}")

단순 토큰 비교: ['이를', '통해', '더', '많은', '정보를', '얻는다']
다어절 구문 비교: ['더', '많은', '정보를', '얻는다']


## 사용자 정의 불용어 파일 읽기

`Path`는 운영체제에 독립적인 상대 경로를 표현하고 `Path.open`은 UTF-8 텍스트 파일을 읽는다. `load_stopwords`는 각 줄의 양끝 공백을 제거한 뒤 빈 줄과 `#` 주석 줄을 제외하고 `set[str]`를 반환한다. `set`은 중복을 제거하고 빠른 포함 검사를 지원한다. 입력 파일 `stopwords-ko.txt`는 노트북과 같은 폴더에 있다.

파일에 다어절 항목이 있으면 앞 셀의 구문 처리처럼 별도 토큰화가 필요하다. 사전 파일도 코드와 함께 버전을 관리해야 학습과 추론 결과를 재현할 수 있다.


In [35]:
def load_stopwords(filepath):
    path = Path(filepath)

    stopword_set = set()

    # encoding="utf-8"을 명시해 운영체제 기본 인코딩이 달라도 한글을 같은 방식으로 읽는다.
    # with 블록이 끝나면 파일 객체가 자동으로 닫힌다.
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            # strip(): 줄 끝 개행과 양끝 공백을 제거해 실제 사전 항목만 남긴다.
            stripped = line.strip()

            # 빈 문자열과 #으로 시작하는 설명 줄은 사전 항목이 아니므로 건너뛴다.
            if not stripped or stripped.startswith("#"):
                continue

            stopword_set.add(stripped)
    return stopword_set

stopword_path = Path("stopwords-ko.txt")
file_stopwords = load_stopwords(stopword_path)

removed_by_file = [
    token
    for token in morpheme_tokens
    if token in file_stopwords
]

# 같은 원본을 순회해 file_stopwords에 포함되지 않은 token만 유지 목록에 추가한다.
kept_by_file = [
    token
    for token in morpheme_tokens
    if token not in file_stopwords
]

print(f"사전 경로: {stopword_path}")
print(f"사전 항목 수: {len(file_stopwords)}")
# set에는 순서가 없으므로 sorted()로 정렬해야 앞부분이 실행할 때마다 같은 순서로 출력된다.
print(f"사전 앞부분: {sorted(file_stopwords)[:5]}")
print(f"파일 사전으로 제거: {removed_by_file}")
print(f"파일 사전 적용 후: {kept_by_file}")

사전 경로: stopwords-ko.txt
사전 항목 수: 595
사전 앞부분: ['가', '가까스로', '가령', '각', '각각']
파일 사전으로 제거: ['이', '에서', '을', '를']
파일 사전 적용 후: ['방법', '은', '특히', '빅데이터', '중요한', '역할', '합니다', '.', '이를', '통해', '더', '많은', '정보', '얻을', '수', '있습니다', '.']


## 전체 파이프라인의 단계별 결과 확인

앞에서 정의한 규칙을 하나의 함수로 연결한다. 반환값은 단계 이름과 중간 결과를 담은 사전이므로 특정 단계에서 정보가 사라졌는지 추적할 수 있다. 각 단계의 분류는 다음과 같다.

1. `raw`: 원문 보존 단계이다.
2. `normalized`: NFC, 도메인 용어 통합, `casefold`를 적용한 정규화 결과이다.
3. `masked`: 이메일과 URL의 실제 값을 가린 민감 정보 정제 결과이다.
4. `tokens`: 문자열을 모델 입력 단위로 나눈 토큰화 선행 단계이다.
5. `filtered`: 부정어와 자리표시자를 보호하면서 불용어를 제거한 정제 결과이다.


In [36]:
model_token_pattern = re.compile(r"__EMAIL__|__URL__|[A-Za-z]+")

def preprocess_text(text):
    raw = text

    # 1단계 정규화: NFC로 유니코드 표현을 맞춘다.
    normalized = unicodedata.normalize("NFC", raw)

    # 2단계 정규화: 앞에서 만든 mapping으로 United Kingdom을 UK로 통합한다.
    normalized = normalize_terms(normalized, term_mapping)

    # 3단계 정규화: casefold()로 UK/uk와 NOT/not의 대소문자를 같은 기준형으로 맞춘다.
    normalized = normalized.casefold()

    # 이메일과 URL은 토큰화 전에 마스킹해야 내부의 점·@·도메인이 여러 토큰으로 흩어지지 않는다.
    # mask_sensitive_text()는 실제 값을 __EMAIL__ 또는 __URL__로 바꾼 새 문자열을 반환한다.
    masked = mask_sensitive_text(normalized)

    # findall()은 model_token_pattern과 일치한 모든 문자열을 원문 순서의 list[str]로 반환한다.
    # 마침표와 @ 같은 패턴 밖의 문자는 tokens에 포함되지 않는다.
    tokens = model_token_pattern.findall(masked)


    filtered = [
        token
        for token in tokens
        if token.startswith("__")
        or token not in sentiment_stopwords
    ]

    return {
        "raw": raw,
        "normalized": normalized,
        "masked": masked,
        "tokens": tokens,
        "filtered": filtered,
    }

pipeline_result = preprocess_text(
    "The United Kingdom support team says the service is NOT bad. "
    "Email master@lifeisgood.com."
)

for stage, value in pipeline_result.items():
    print(f"{stage}: {value}")

# 필터링 전후 list 길이를 비교해 불용어 정제로 몇 개의 토큰이 줄었는지 확인한다.
print(
    "토큰 수 변화: "
    f"{len(pipeline_result['tokens'])} -> "
    f"{len(pipeline_result['filtered'])}"
)


raw: The United Kingdom support team says the service is NOT bad. Email master@lifeisgood.com.
normalized: the uk support team says the service is not bad. email master@lifeisgood.com.
masked: the uk support team says the service is not bad. email __EMAIL__.
tokens: ['the', 'uk', 'support', 'team', 'says', 'the', 'service', 'is', 'not', 'bad', 'email', '__EMAIL__']
filtered: ['uk', 'support', 'team', 'says', 'service', 'not', 'bad', 'email', '__EMAIL__']
토큰 수 변화: 12 -> 9


## 정리와 실무 점검 기준

- 정규화는 분석 목적상 같은 것으로 볼 표기를 기준형으로 맞추는 작업이고, 정제는 불필요·오류·민감 정보를 제거하거나 마스킹하는 작업이다.
- 사용한 함수가 아니라 처리 목적과 보존하려는 정보를 기준으로 두 작업을 구분한다.
- 토큰화와 형태소 분석은 정제·정규화가 아니라 두 작업의 입력 단위를 만드는 선행 단계이다.
- 원문을 별도 보관하고 단계별 결과와 규칙 버전을 남겨야 정보 손실을 추적할 수 있다.
- 표기 정규화와 민감 정보 처리는 보통 토큰화 전에 수행하고, 빈도·길이·불용어 필터는 토큰화 뒤에 수행한다.
- 희귀 단어 기준과 불용어 사전은 학습 데이터에서만 만들고 검증·테스트 데이터에는 그대로 적용한다.
- 감성 분석의 부정어, 개체명 인식의 대소문자, 보안 분석의 URL처럼 과업 신호가 되는 정보는 보호해야 한다.
- 제거율, 대표 전후 샘플, 핵심 용어 보존 여부를 자동 테스트해 과도한 정제를 조기에 발견한다.
- KoNLPy와 NLTK 리소스는 준비 셀에서 직접 설치·다운로드한 뒤 학습과 추론에서 같은 버전을 사용한다.
